In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Load the FAQ Dataset


In [2]:
import pandas as pd
import numpy as np
import time

from utils.cache.faq_data_container import FAQDataContainer

faq_data = FAQDataContainer()
faq_df = faq_data.faq_df
test_df = faq_data.test_df

Loaded 8 FAQ entries
Loaded 80 test queries


In [3]:
faq_df.head().style

,id,question,answer
0,0,How do I get a refund?,"To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days."
1,1,Can I reset my password?,Click **Forgot Password** on the login page and follow the email instructions. Check your spam folder if you don't see the email.
2,2,Where is my order?,Use the tracking link sent to your email after shipping. Orders typically arrive within 2-7 business days depending on your location.
3,3,How long is the warranty?,All electronic products include a 12-month warranty from the purchase date. Extended warranties are available for purchase.
4,4,Do you ship internationally?,"Yes, we ship to over 50 countries worldwide. International shipping fees and delivery times vary by destination."


### Implementation With Redis

In [4]:
import os

REDIS_URL = os.getenv("REDIS_URL")

In [5]:
import redis

try:
    r = redis.Redis.from_url(REDIS_URL)
    r.ping()
    print("Redis is running and accessible")
except redis.ConnectionError:
    print(" Cannot connect to Redis")
    raise

Redis is running and accessible


### Using a Cache-Optimized Embedding Model (langcache-embed-v1)
https://huggingface.co/redis/langcache-embed-v1

In [6]:
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

langcache_embed = HFTextVectorizer(
    model="redis/langcache-embed-v1",
    cache=EmbeddingsCache(redis_client=r, ttl=3600)
)

09:32:14 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu
09:32:14 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


### Create the Redis Semantic Cache

In [7]:
from redisvl.extensions.cache.llm import SemanticCache

cache = SemanticCache(
    name="faq-cache",
    vectorizer=langcache_embed,
    redis_client=r,
    distance_threshold=0.3
)

09:32:21 redisvl.index.index INFO   Index already exists, not overwriting.


### Load the Cache with FAQ Data

In [8]:
for i in range(len(faq_df)):
    cache.store(
        prompt=faq_df.iloc[i]["question"],
        response=faq_df.iloc[i]["answer"]
    )

In [9]:
result = cache.check("I need a refund for my purchase")

In [10]:
result

[{'entry_id': '60fd55b8527fcd2bf427d81dc3f4c47c4bf8904c9802ffecbcf2c02b38f537ac',
  'prompt': 'How do I get a refund?',
  'response': 'To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days.',
  'vector_distance': 0.256070017815,
  'inserted_at': 1781323341.35,
  'updated_at': 1781323341.35,
  'key': 'faq-cache:60fd55b8527fcd2bf427d81dc3f4c47c4bf8904c9802ffecbcf2c02b38f537ac'}]

### Implement TTL (time-to-live) policy to keep cache fresh

In [11]:
cache.set_ttl(86400)

## End-to-End LLM Example


In [12]:
from utils.cache.config import load_openai_key
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_openai_key()

MODEL_NAME = "gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.1,
    max_tokens=150,
)

> OpenAI API key is already loaded in the environment


In [13]:
def get_llm_response(question: str) -> str:
    prompt = f"""
    You are a helpful customer support assistant. Answer this customer question concisely and professionally:
    
    Question: {question}
    
    Provide a helpful response in 1-2 sentences. If you don't have specific information, give a general helpful response.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [14]:
from utils.cache.evals import PerfEval

perf_eval = PerfEval()

test_questions = [
    "How can I get my money back?",
    "I want a refund please",
    "What's your return policy?",
    "I forgot my password",
    "Can you help me reset my password?",
    "What are your shipping costs?",
    "Do you offer installation services?",
    "Can I schedule a phone call with support?",
    "How do I cancel my subscription?",
    "How much does shipping cost?",
    "I need to cancel my account",
]

perf_eval.set_total_queries(len(test_questions))

In [15]:
with perf_eval:
    for i, question in enumerate(test_questions, 1):
        print(f"\n[{i}] Question: '{question}'")

        perf_eval.start()

        if cached_result := cache.check(question):
            # Cache HIT
            perf_eval.tick("cache_hit")
            print(
                f"    CACHE HIT (distance: {cached_result[0]['vector_distance']:.3f})"
            )
            print(f"     Cached question: {cached_result[0]['prompt'][:80]}...")
            print(f"     Cached response: {cached_result[0]['response'][:80]}...")
        else:
            # Cache MISS - call LLM
            perf_eval.tick("cache_miss")  # Time for cache check
            print(f"    CACHE MISS")
            print(f"    Calling LLM... ", end="")

            # Call LLM and track the call
            perf_eval.start()
            llm_response = get_llm_response(question)
            perf_eval.tick("llm_call")
            perf_eval.record_llm_call(MODEL_NAME, question, llm_response)
            print(f"     LLM response: {llm_response[:80]}...")
            cache.store(prompt=question, response=llm_response)


[1] Question: 'How can I get my money back?'
    CACHE HIT (distance: 0.249)
     Cached question: How do I get a refund?...
     Cached response: To request a refund, visit your orders page and select **Request Refund**. Refun...

[2] Question: 'I want a refund please'
    CACHE HIT (distance: 0.161)
     Cached question: How do I get a refund?...
     Cached response: To request a refund, visit your orders page and select **Request Refund**. Refun...

[3] Question: 'What's your return policy?'
    CACHE MISS
    Calling LLM... 09:32:27 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
     LLM response: Our return policy allows you to return items within 30 days of purchase for a fu...

[4] Question: 'I forgot my password'
    CACHE HIT (distance: 0.184)
     Cached question: Can I reset my password?...
     Cached response: Click **Forgot Password** on the login page and follow the email instructions. C...

[5] Question: 'Can you help me r

In [16]:
np.mean(perf_eval.durations_by_label['cache_hit'])

np.float64(0.1459829466683524)

In [17]:
np.mean(perf_eval.durations_by_label['llm_call'])

np.float64(1.892707109451294)